In [1]:
!nvidia-smi

Wed Sep  2 10:17:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
!/content/venv/bin/python --version

Python 3.10.12


In [7]:
!/content/venv/bin/python -m pip install -q \
"vllm==0.6.*" \
"transformers==4.46.*" \
"accelerate==1.1.*" \
"autoawq==0.2.*" \
"httpx==0.27.*" \
"openai==1.54.*"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [32]:
import os
import signal
import subprocess

PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args):
    cmd = ["/content/venv/bin/python",
           "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args=None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))

    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 34150, logging to /content/server.log


In [11]:
SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,        # bare flag
    "--tool-call-parser": "hermes",           # Qwen2.5 family uses hermes
}
server = launch_server(SERVER_ARGS)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 13110, logging to /content/server.log


In [12]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader
# note the number; compare to yesterday's fp16 resident VRAM

11723 MiB


In [13]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-language=ru (Japan time zone ID 891)
```

Replace `[y

In [15]:
from openai import OpenAI

# Two tools the model may call. Shapes match the OpenAI tools schema.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

# The 3 canonical prompts. Each carries how many attempts (k) it gets and how many
# tool calls a correct answer makes. n = sum of k = 10.
#   two_tool:   needs BOTH tools (weather + calculator)   -> expect >= 1 call
#   single:     needs ONE tool                            -> expect >= 1 call
#   distractor: needs NO tool, must NOT call one          -> expect 0 calls
CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]


def _tool_calls_of(message) -> list:
    """Return the parsed tool_calls list on a response message, or []."""
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []


def _valid_call(call) -> bool:
    """A tool call is valid if it names a known function and its arguments
    parse as JSON with the required field present."""
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False


def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    """Run the smoke test. Returns a result dict with counts and the pass gate."""
    client = OpenAI(base_url=base_url, api_key="not-needed")

    total_attempts = 0
    valid_call_attempts = 0          # attempts that returned >=1 valid tool call
    distractor_attempts = 0
    distractor_call_free = 0         # distractor attempts that made NO tool call
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                # a "wants a call" prompt counts toward the 8/10 gate when it
                # returns at least one valid tool call
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                # the distractor counts toward the 8/10 gate when it correctly
                # makes NO tool call, and separately toward distractor compliance
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    # gate: >=8/10 correct behaviours AND distractor call-free in the majority
    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,       # 10
        "score": valid_call_attempts,           # correct behaviours, out of 10
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }

# paste the contents of smoke_test.py, then:
result = run_smoke(base_url="http://localhost:8000/v1",
                   model="Qwen/Qwen2.5-1.5B-Instruct-AWQ")
print(result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [17]:
import json
with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

In [21]:
%%writefile model-lock.md
# Model lock (team record)

## The locked model

- Model id: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Quantisation: awq
- Why this one: Passed the smoke test with 10/10 and uses less VRAM than the FP16 model.

## The launch flags

```text
--model Qwen/Qwen2.5-1.5B-Instruct-AWQ
--dtype half
--max-model-len 4096
--gpu-memory-utilization 0.85
--port 8000
--quantization awq
--enable-auto-tool-choice
--tool-call-parser hermes

Writing model-lock.md


In [22]:
!python verify_cell\[1\].py

smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
